In [3]:
"""
AFAD Haber Scraper — Playwright tabanlı (v3)
Çıktı: afad_haberler.csv
 
Jupyter'da çalıştırmak için son hücreye:
    await main()
 
Terminal'de çalıştırmak için:
    python afad_scraper.py
"""
 
import asyncio
import csv
import logging
import random
from datetime import datetime
 
from playwright.async_api import async_playwright, Page, TimeoutError as PWTimeout
 
# ─── Ayarlar ────────────────────────────────────────────────────────────────
LIST_URL    = "https://afad.gov.tr/haberler"
OUTPUT_FILE = "afad_haberler.csv"
DELAY_MIN   = 1.2
DELAY_MAX   = 2.8
HEADLESS    = True   # False yaparsanız tarayıcı görünür açılır (debug için)
 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
 
 
# ─── Yardımcı ────────────────────────────────────────────────────────────────
 
async def _loc_text(parent, selector: str) -> str:
    """Locator içindeki ilk eşleşen elementin metnini döndürür."""
    try:
        el = parent.locator(selector).first
        if await el.count():
            return (await el.inner_text()).strip()
    except Exception:
        pass
    return ""
 
 
# ─── Ana liste sayfası: scroll + kart toplama ────────────────────────────────
 
async def collect_cards(page: Page) -> list[dict]:
    log.info("Ana sayfa yükleniyor…")
 
    # domcontentloaded: AFAD ajax yükleme yaptığı için networkidle donuyor
    await page.goto(LIST_URL, wait_until="domcontentloaded", timeout=60_000)
    await asyncio.sleep(3)
 
    MAX_SCROLLS = 10
 
    for scroll_i in range(1, MAX_SCROLLS + 1):
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await asyncio.sleep(2.5)
 
        curr_count = await page.locator("a.card.news-card-horizontal").count()
        log.info(f"  Scroll {scroll_i}/{MAX_SCROLLS} — kart sayısı: {curr_count}")
 
        # "Daha fazla" / load-more butonlarını dene
        for btn_sel in [
            '[id^="wf87acb"]',
            'button:has-text("Daha Fazla")',
            'button:has-text("Daha fazla")',
            'a:has-text("Daha Fazla")',
            '.load-more',
        ]:
            try:
                btn = page.locator(btn_sel).first
                if await btn.is_visible(timeout=800):
                    await btn.scroll_into_view_if_needed()
                    await btn.click()
                    log.info(f"  Butona tıklandı: {btn_sel}")
                    await asyncio.sleep(2.5)
                    break
            except Exception:
                pass
 
    log.info("  10 scroll tamamlandı.")
 
    # ── Kartları parse et ────────────────────────────────────────────────────
    card_els = await page.locator("a.card.news-card-horizontal").all()
    log.info(f"Toplam {len(card_els)} kart bulundu.")
 
    items = []
    for card in card_els:
        href = await card.get_attribute("href") or ""
        if href.startswith("//"):
            href = "https:" + href
        elif href.startswith("/"):
            href = "https://afad.gov.tr" + href
 
        title  = await _loc_text(card, "h5.card-title")
        date   = await _loc_text(card, "span.cardDate")
        teaser = await _loc_text(card, "p.card-text")
 
        if href:
            items.append({"href": href, "title": title, "date": date, "teaser": teaser})
 
    return items
 
 
# ─── Detay sayfası ───────────────────────────────────────────────────────────
 
async def scrape_detail(page: Page, url: str) -> dict:
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=40_000)
        await asyncio.sleep(1.5)
    except PWTimeout:
        log.warning(f"  Timeout: {url}")
        return {"detail_title": "", "full_text": ""}
 
    # ── Başlık ───────────────────────────────────────────────────────────────
    detail_title = ""
    for sel in [
        '[id^="w3152ee266cf947b5b89d50c1da9b2ab0"] h2 span',
        "h1 span", "h2 span", "h1", "h2",
    ]:
        try:
            el = page.locator(sel).first
            if await el.count():
                detail_title = (await el.inner_text(timeout=2_000)).strip()
                if detail_title:
                    break
        except Exception:
            pass
 
    # ── İçerik alanını bul ───────────────────────────────────────────────────
    content_sel = None
    for sel in [
        '[id$="2"] > div',
        ".news-detail-content",
        ".haber-detay-content",
        "article .content",
        ".detail-content",
        "main .col-md-8",
        "main .col-lg-8",
    ]:
        try:
            el = page.locator(sel).first
            if await el.count():
                content_sel = sel
                break
        except Exception:
            pass
 
    # ── Paragrafları topla ───────────────────────────────────────────────────
    # NOT: Locator nesnelerinde .query_selector_all() ÇALIŞMAZ
    # Doğru kullanım: .locator().count() + .locator().nth(i)
    paragraphs = []
 
    if content_sel:
        content = page.locator(content_sel).first
 
        # Önce span > span > span yapısını dene
        spans      = content.locator("p span span span")
        span_count = await spans.count()
 
        if span_count:
            for j in range(span_count):
                t = (await spans.nth(j).inner_text()).strip()
                if t:
                    paragraphs.append(t)
        else:
            # Düz <p> etiketlerini çek
            ps       = content.locator("p")
            ps_count = await ps.count()
            for j in range(ps_count):
                t = (await ps.nth(j).inner_text()).strip()
                if t:
                    paragraphs.append(t)
 
    # Son çare: tüm sayfadan <p> çek
    if not paragraphs:
        ps       = page.locator("main p, article p, .container p")
        ps_count = await ps.count()
        for j in range(ps_count):
            t = (await ps.nth(j).inner_text()).strip()
            if t and len(t) > 30:   # çok kısa parçaları atla
                paragraphs.append(t)
 
    full_text = "\n\n".join(paragraphs)
    return {"detail_title": detail_title, "full_text": full_text}
 
 
# ─── Ana akış ────────────────────────────────────────────────────────────────
 
async def main():
    log.info("AFAD scraper başlatılıyor…")
    start_time = datetime.now()
 
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS)
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            ),
            locale="tr-TR",
        )
        page = await context.new_page()
 
        # 1. Ana sayfadan tüm kartları topla
        cards = await collect_cards(page)
 
        fieldnames = ["date", "title", "teaser", "url", "detail_title", "full_text"]
 
        with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
 
            # 2. Her kartın detay sayfasına gir
            for i, card in enumerate(cards, 1):
                log.info(f"[{i}/{len(cards)}] {card['title'][:60]}…")
                await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
 
                detail = await scrape_detail(page, card["href"])
 
                writer.writerow({
                    "date"         : card["date"],
                    "title"        : card["title"],
                    "teaser"       : card["teaser"],
                    "url"          : card["href"],
                    "detail_title" : detail["detail_title"],
                    "full_text"    : detail["full_text"],
                })
                f.flush()   # anlık kaydet, çökerse veri gitmesins
# Terminal'de çalıştırmak için:
#     python afad_scraper.py
 
await main()

18:38:35  INFO  AFAD scraper başlatılıyor…
18:38:36  INFO  Ana sayfa yükleniyor…
18:38:42  INFO    Scroll 1/10 — kart sayısı: 8
18:38:42  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:38:47  INFO    Scroll 2/10 — kart sayısı: 16
18:38:47  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:38:52  INFO    Scroll 3/10 — kart sayısı: 24
18:38:53  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:38:58  INFO    Scroll 4/10 — kart sayısı: 32
18:38:58  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:39:03  INFO    Scroll 5/10 — kart sayısı: 40
18:39:03  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:39:08  INFO    Scroll 6/10 — kart sayısı: 48
18:39:08  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:39:13  INFO    Scroll 7/10 — kart sayısı: 56
18:39:13  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:39:18  INFO    Scroll 8/10 — kart sayısı: 64
18:39:18  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:39:23  INFO    Scroll 9/10 — kart sayısı: 72
18:39:23

In [8]:
"""
Valilik Haber Scraper — Hatay / Adıyaman / Kahramanmaraş
Playwright tabanlı, scroll-based sayfa

Jupyter'da:   await main()
Terminalde:   python valilik_scraper.py
"""

import asyncio
import csv
import logging
import random
from datetime import datetime

from playwright.async_api import async_playwright, Page, TimeoutError as PWTimeout

# ─── Ayarlar ────────────────────────────────────────────────────────────────

SITES = [
    {
        "name"       : "hatay",
        "url"        : "https://www.hatay.gov.tr/haberler",
        "max_scrolls": 45,
        "output"     : "hatay_haberler.csv",
    },
    {
        "name"       : "adiyaman",
        "url"        : "https://www.adiyaman.gov.tr/haberler",
        "max_scrolls": 30,
        "output"     : "adiyaman_haberler.csv",
    },
    {
        "name"       : "kahramanmaras",
        "url"        : "https://www.kahramanmaras.gov.tr/haberler",
        "max_scrolls": 30,
        "output"     : "kahramanmaras_haberler.csv",
    },
]

DELAY_MIN = 0.8
DELAY_MAX = 1.6
HEADLESS  = True

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ─── Yardımcı ────────────────────────────────────────────────────────────────

async def _loc_text(parent, selector: str) -> str:
    try:
        el = parent.locator(selector).first
        if await el.count():
            return (await el.inner_text()).strip()
    except Exception:
        pass
    return ""


async def safe_goto(page: Page, url: str, retries: int = 3) -> bool:
    """Sayfayı yükle, başarısız olursa tekrar dene."""
    for attempt in range(1, retries + 1):
        try:
            # Önce "load" dene, timeout gelirse "commit" ile devam et
            await page.goto(url, wait_until="load", timeout=45_000)
            return True
        except Exception as e:
            log.warning(f"  Deneme {attempt}/{retries} başarısız: {e}")
            if attempt < retries:
                await asyncio.sleep(random.uniform(3, 6))
    return False


# ─── Scroll + kart toplama ───────────────────────────────────────────────────

async def collect_cards(page: Page, site: dict) -> list[dict]:
    log.info(f"[{site['name']}] Ana sayfa yükleniyor: {site['url']}")

    ok = await safe_goto(page, site["url"])
    if not ok:
        log.error(f"[{site['name']}] Sayfa yüklenemedi, atlanıyor.")
        return []

    await asyncio.sleep(3)

    for i in range(1, site["max_scrolls"] + 1):
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await asyncio.sleep(2.0)

        curr_count = await page.locator("a.card.news-card-horizontal").count()
        log.info(f"  Scroll {i}/{site['max_scrolls']} — kart: {curr_count}")

        for btn_sel in [
            '[id^="wf87acb"]',
            'button:has-text("Daha Fazla")',
            'button:has-text("Daha fazla")',
            'a:has-text("Daha Fazla")',
            '.load-more',
        ]:
            try:
                btn = page.locator(btn_sel).first
                if await btn.is_visible(timeout=500):
                    await btn.scroll_into_view_if_needed()
                    await btn.click()
                    log.info(f"  Butona tıklandı: {btn_sel}")
                    await asyncio.sleep(2.0)
                    break
            except Exception:
                pass

    card_els = await page.locator("a.card.news-card-horizontal").all()
    log.info(f"[{site['name']}] Toplam {len(card_els)} kart bulundu.")

    items = []
    for card in card_els:
        href = await card.get_attribute("href") or ""
        if href.startswith("//"):
            href = "https:" + href
        elif href.startswith("/"):
            base = site["url"].split("/haberler")[0]
            href = base + href

        title  = await _loc_text(card, "h5.card-title")
        date   = await _loc_text(card, "span.cardDate")
        teaser = await _loc_text(card, "p.card-text")

        if href:
            items.append({"href": href, "title": title, "date": date, "teaser": teaser})

    return items


# ─── Detay sayfası ───────────────────────────────────────────────────────────

async def scrape_detail(page: Page, url: str) -> dict:
    ok = await safe_goto(page, url)
    if not ok:
        return {"detail_title": "", "full_text": ""}

    await asyncio.sleep(1.0)

    # ── Başlık ───────────────────────────────────────────────────────────────
    detail_title = ""
    for sel in [
        '[id^="w3152ee"] h2 span',
        '[id^="w6e94dc"] h2 span',
        "h1 span", "h2 span", "h1", "h2",
    ]:
        try:
            el = page.locator(sel).first
            if await el.count():
                detail_title = (await el.inner_text(timeout=2_000)).strip()
                if detail_title:
                    break
        except Exception:
            pass

    # ── İçerik alanı ─────────────────────────────────────────────────────────
    content_sel = None
    for sel in [
        '[id^="w6e94dc"] > div',
        '[id$="2"] > div',
        ".news-detail-content",
        ".haber-detay-content",
        "article .content",
        ".detail-content",
        "main .col-md-8",
        "main .col-lg-8",
    ]:
        try:
            el = page.locator(sel).first
            if await el.count():
                content_sel = sel
                break
        except Exception:
            pass

    # ── Paragrafları topla ───────────────────────────────────────────────────
    paragraphs = []

    if content_sel:
        content = page.locator(content_sel).first

        spans      = content.locator("p span")
        span_count = await spans.count()

        if span_count:
            for j in range(span_count):
                t = (await spans.nth(j).inner_text()).strip()
                if t:
                    paragraphs.append(t)
        else:
            ps       = content.locator("p")
            ps_count = await ps.count()
            for j in range(ps_count):
                t = (await ps.nth(j).inner_text()).strip()
                if t:
                    paragraphs.append(t)

    if not paragraphs:
        ps       = page.locator("main p, article p, .container p")
        ps_count = await ps.count()
        for j in range(ps_count):
            t = (await ps.nth(j).inner_text()).strip()
            if t and len(t) > 30:
                paragraphs.append(t)

    seen, clean = set(), []
    for p in paragraphs:
        if p not in seen:
            seen.add(p)
            clean.append(p)

    return {"detail_title": detail_title, "full_text": "\n\n".join(clean)}


# ─── Tek site işle ───────────────────────────────────────────────────────────

async def process_site(page: Page, site: dict):
    log.info(f"\n{'='*50}\n  {site['name'].upper()} başlıyor\n{'='*50}")

    cards = await collect_cards(page, site)
    if not cards:
        return

    fieldnames = ["date", "title", "teaser", "url", "detail_title", "full_text"]

    with open(site["output"], "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for i, card in enumerate(cards, 1):
            log.info(f"[{site['name']}] [{i}/{len(cards)}] {card['title'][:55]}…")
            await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

            detail = await scrape_detail(page, card["href"])

            writer.writerow({
                "date"         : card["date"],
                "title"        : card["title"],
                "teaser"       : card["teaser"],
                "url"          : card["href"],
                "detail_title" : detail["detail_title"],
                "full_text"    : detail["full_text"],
            })
            f.flush()

    log.info(f"[{site['name']}] Tamamlandı → {site['output']}")


# ─── Ana akış ────────────────────────────────────────────────────────────────

async def main():
    log.info("Valilik scraper başlatılıyor…")
    start_time = datetime.now()

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=HEADLESS,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-sandbox",
                "--disable-dev-shm-usage",
            ],
        )
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/123.0.0.0 Safari/537.36"
            ),
            locale="tr-TR",
            timezone_id="Europe/Istanbul",
            viewport={"width": 1280, "height": 900},
            # Gerçek tarayıcı gibi görünmek için
            extra_http_headers={
                "Accept-Language": "tr-TR,tr;q=0.9,en-US;q=0.8,en;q=0.7",
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            },
        )

        # navigator.webdriver = false yap (bot tespitini atla)
        await context.add_init_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )

        page = await context.new_page()

        for site in SITES:
            await process_site(page, site)

        await browser.close()

    elapsed = datetime.now() - start_time
    log.info(f"\nHepsi tamamlandı! Toplam süre: {elapsed}")


# Jupyter'da:  await main()
# Terminalde:  python valilik_scraper.py
await main()

12:01:00  INFO  Valilik scraper başlatılıyor…
12:01:01  INFO  
  HATAY başlıyor
12:01:01  INFO  [hatay] Ana sayfa yükleniyor: https://www.hatay.gov.tr/haberler
12:01:20  INFO    Scroll 1/45 — kart: 10
12:01:20  INFO    Butona tıklandı: a:has-text("Daha Fazla")
12:01:24  INFO    Scroll 2/45 — kart: 20
12:01:24  INFO    Butona tıklandı: a:has-text("Daha Fazla")
12:01:28  INFO    Scroll 3/45 — kart: 30
12:01:28  INFO    Butona tıklandı: a:has-text("Daha Fazla")
12:01:33  INFO    Scroll 4/45 — kart: 40
12:01:33  INFO    Butona tıklandı: a:has-text("Daha Fazla")
12:01:37  INFO    Scroll 5/45 — kart: 50
12:01:37  INFO    Butona tıklandı: a:has-text("Daha Fazla")
12:01:41  INFO    Scroll 6/45 — kart: 60
12:01:41  INFO    Butona tıklandı: a:has-text("Daha Fazla")
12:01:45  INFO    Scroll 7/45 — kart: 70
12:01:45  INFO    Butona tıklandı: a:has-text("Daha Fazla")
12:01:49  INFO    Scroll 8/45 — kart: 80
12:01:50  INFO    Butona tıklandı: a:has-text("Daha Fazla")
12:01:54  INFO    Scroll 9/45 — 

CancelledError: 

In [9]:
"""
Adıyaman Valiliği Haber Scraper
https://www.adiyaman.gov.tr/haberler

Jupyter'da:   await main()
Terminalde:   python adiyaman_scraper.py
"""

import asyncio
import csv
import logging
import random
from datetime import datetime

from playwright.async_api import async_playwright, Page, TimeoutError as PWTimeout

# ─── Ayarlar ────────────────────────────────────────────────────────────────
LIST_URL    = "https://www.adiyaman.gov.tr/haberler"
OUTPUT_FILE = "adiyaman_haberler.csv"
MAX_SCROLLS = 6      # 55 haber, her scroll ~10 kart = 6 yeterli
DELAY_MIN   = 0.8
DELAY_MAX   = 1.6
HEADLESS    = True

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ─── Yardımcı ────────────────────────────────────────────────────────────────

async def _loc_text(parent, selector: str) -> str:
    try:
        el = parent.locator(selector).first
        if await el.count():
            return (await el.inner_text()).strip()
    except Exception:
        pass
    return ""


async def safe_goto(page: Page, url: str, retries: int = 3) -> bool:
    for attempt in range(1, retries + 1):
        try:
            await page.goto(url, wait_until="load", timeout=45_000)
            return True
        except Exception as e:
            log.warning(f"  Deneme {attempt}/{retries} başarısız: {e}")
            if attempt < retries:
                await asyncio.sleep(random.uniform(3, 6))
    return False


# ─── Scroll + kart toplama ───────────────────────────────────────────────────

async def collect_cards(page: Page) -> list[dict]:
    log.info(f"Ana sayfa yükleniyor: {LIST_URL}")

    ok = await safe_goto(page, LIST_URL)
    if not ok:
        log.error("Sayfa yüklenemedi.")
        return []

    await asyncio.sleep(3)

    for i in range(1, MAX_SCROLLS + 1):
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await asyncio.sleep(2.0)

        curr_count = await page.locator("a.card.news-card-horizontal").count()
        log.info(f"  Scroll {i}/{MAX_SCROLLS} — kart: {curr_count}")

        # "Daha fazla" butonlarını dene
        for btn_sel in [
            '[id^="wf87acb"]',
            'button:has-text("Daha Fazla")',
            'button:has-text("Daha fazla")',
            'a:has-text("Daha Fazla")',
            '.load-more',
        ]:
            try:
                btn = page.locator(btn_sel).first
                if await btn.is_visible(timeout=500):
                    await btn.scroll_into_view_if_needed()
                    await btn.click()
                    log.info(f"  Butona tıklandı: {btn_sel}")
                    await asyncio.sleep(2.0)
                    break
            except Exception:
                pass

    card_els = await page.locator("a.card.news-card-horizontal").all()
    log.info(f"Toplam {len(card_els)} kart bulundu.")

    items = []
    for card in card_els:
        href = await card.get_attribute("href") or ""
        if href.startswith("//"):
            href = "https:" + href
        elif href.startswith("/"):
            href = "https://www.adiyaman.gov.tr" + href

        title  = await _loc_text(card, "h5.card-title")
        date   = await _loc_text(card, "span.cardDate")
        teaser = await _loc_text(card, "p.card-text")

        if href:
            items.append({"href": href, "title": title, "date": date, "teaser": teaser})

    return items


# ─── Detay sayfası ───────────────────────────────────────────────────────────

async def scrape_detail(page: Page, url: str) -> dict:
    ok = await safe_goto(page, url)
    if not ok:
        return {"detail_title": "", "full_text": ""}

    await asyncio.sleep(1.0)

    # ── Başlık ───────────────────────────────────────────────────────────────
    detail_title = ""
    for sel in ["h1 span", "h2 span", "h1", "h2"]:
        try:
            el = page.locator(sel).first
            if await el.count():
                detail_title = (await el.inner_text(timeout=2_000)).strip()
                if detail_title:
                    break
        except Exception:
            pass

    # ── İçerik alanı ─────────────────────────────────────────────────────────
    # Adıyaman: id^="w9727509..." > div altında div[N] > div yapısı
    # Her sayfada id değişiyor, pattern ile buluyoruz
    content_el = None
    for sel in [
        '[id^="w9727509"] > div',
        '[id$="2"] > div',
        ".news-detail-content",
        ".haber-detay-content",
        "article .content",
        ".detail-content",
        "main .col-md-8",
        "main .col-lg-8",
    ]:
        try:
            el = page.locator(sel).first
            if await el.count():
                content_el = el
                break
        except Exception:
            pass

    # ── Paragrafları topla ───────────────────────────────────────────────────
    # Adıyaman yapısı: content > div[N] > div > text()
    # Normal p tag değil, div içinde text var
    paragraphs = []

    if content_el:
        # Önce div > div yapısını dene (Adıyaman'a özgü)
        inner_divs      = content_el.locator("div > div")
        inner_div_count = await inner_divs.count()

        if inner_div_count:
            for j in range(inner_div_count):
                t = (await inner_divs.nth(j).inner_text()).strip()
                if t and len(t) > 10:
                    paragraphs.append(t)
        else:
            # p tag'leri dene
            ps       = content_el.locator("p")
            ps_count = await ps.count()
            for j in range(ps_count):
                t = (await ps.nth(j).inner_text()).strip()
                if t:
                    paragraphs.append(t)

    # Son çare: tüm sayfadan topla
    if not paragraphs:
        ps       = page.locator("main p, article p, .container p, main div > div")
        ps_count = await ps.count()
        for j in range(ps_count):
            t = (await ps.nth(j).inner_text()).strip()
            if t and len(t) > 30:
                paragraphs.append(t)

    # Tekrar edenleri temizle
    seen, clean = set(), []
    for p in paragraphs:
        if p not in seen:
            seen.add(p)
            clean.append(p)

    return {"detail_title": detail_title, "full_text": "\n\n".join(clean)}


# ─── Ana akış ────────────────────────────────────────────────────────────────

async def main():
    log.info("Adıyaman scraper başlatılıyor…")
    start_time = datetime.now()

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=HEADLESS,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-sandbox",
                "--disable-dev-shm-usage",
            ],
        )
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/123.0.0.0 Safari/537.36"
            ),
            locale="tr-TR",
            timezone_id="Europe/Istanbul",
            viewport={"width": 1280, "height": 900},
            extra_http_headers={
                "Accept-Language": "tr-TR,tr;q=0.9,en-US;q=0.8",
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            },
        )
        await context.add_init_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )

        page = await context.new_page()

        cards = await collect_cards(page)

        fieldnames = ["date", "title", "teaser", "url", "detail_title", "full_text"]

        with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

            for i, card in enumerate(cards, 1):
                log.info(f"[{i}/{len(cards)}] {card['title'][:60]}…")
                await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

                detail = await scrape_detail(page, card["href"])

                writer.writerow({
                    "date"         : card["date"],
                    "title"        : card["title"],
                    "teaser"       : card["teaser"],
                    "url"          : card["href"],
                    "detail_title" : detail["detail_title"],
                    "full_text"    : detail["full_text"],
                })
                f.flush()

        await browser.close()

    elapsed = datetime.now() - start_time
    log.info(f"Tamamlandı! {len(cards)} haber — Süre: {elapsed}")
    log.info(f"Çıktı: {OUTPUT_FILE}")


# Jupyter'da:  await main()
# Terminalde:  python adiyaman_scraper.py
await main()

18:52:19  INFO  Adıyaman scraper başlatılıyor…
18:52:19  INFO  Ana sayfa yükleniyor: https://www.adiyaman.gov.tr/haberler
18:52:28  INFO    Scroll 1/6 — kart: 10
18:52:29  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:52:33  INFO    Scroll 2/6 — kart: 20
18:52:33  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:52:37  INFO    Scroll 3/6 — kart: 30
18:52:37  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:52:41  INFO    Scroll 4/6 — kart: 40
18:52:41  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:52:45  INFO    Scroll 5/6 — kart: 50
18:52:45  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:52:49  INFO    Scroll 6/6 — kart: 60
18:52:50  INFO    Butona tıklandı: a:has-text("Daha Fazla")
18:52:52  INFO  Toplam 70 kart bulundu.
18:52:52  INFO  [1/70] 2025 Yılı Genel Değerlendirme Toplantısı…
18:52:55  INFO  [2/70] Valimiz Sayın Dr. Osman Varol’un “10 Ocak Çalışan Gazetecile…
18:52:58  INFO  [3/70] Valimiz Sayın Dr. Osman Varol’un “3 Aralık Dünya Engelliler …
18:

In [10]:
"""
Kahramanmaraş Valiliği Haber Scraper
https://www.kahramanmaras.gov.tr/haberler
715 haber, 70 scroll

Jupyter'da:   await main()
Terminalde:   python kahramanmaras_scraper.py
"""

import asyncio
import csv
import logging
import random
from datetime import datetime

from playwright.async_api import async_playwright, Page, TimeoutError as PWTimeout

# ─── Ayarlar ────────────────────────────────────────────────────────────────
LIST_URL    = "https://www.kahramanmaras.gov.tr/haberler"
OUTPUT_FILE = "kahramanmaras_haberler.csv"
MAX_SCROLLS = 70
DELAY_MIN   = 0.8
DELAY_MAX   = 1.6
HEADLESS    = True

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ─── Yardımcı ────────────────────────────────────────────────────────────────

async def _loc_text(parent, selector: str) -> str:
    try:
        el = parent.locator(selector).first
        if await el.count():
            return (await el.inner_text()).strip()
    except Exception:
        pass
    return ""


async def safe_goto(page: Page, url: str, retries: int = 3) -> bool:
    for attempt in range(1, retries + 1):
        try:
            await page.goto(url, wait_until="load", timeout=45_000)
            return True
        except Exception as e:
            log.warning(f"  Deneme {attempt}/{retries} başarısız: {e}")
            if attempt < retries:
                await asyncio.sleep(random.uniform(3, 6))
    return False


# ─── Scroll + kart toplama ───────────────────────────────────────────────────

async def collect_cards(page: Page) -> list[dict]:
    log.info(f"Ana sayfa yükleniyor: {LIST_URL}")

    ok = await safe_goto(page, LIST_URL)
    if not ok:
        log.error("Sayfa yüklenemedi.")
        return []

    await asyncio.sleep(3)

    for i in range(1, MAX_SCROLLS + 1):
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await asyncio.sleep(2.0)

        curr_count = await page.locator("a.card.news-card-horizontal").count()
        log.info(f"  Scroll {i}/{MAX_SCROLLS} — kart: {curr_count}")

        for btn_sel in [
            '[id^="wf87acb"]',
            'button:has-text("Daha Fazla")',
            'button:has-text("Daha fazla")',
            'a:has-text("Daha Fazla")',
            '.load-more',
        ]:
            try:
                btn = page.locator(btn_sel).first
                if await btn.is_visible(timeout=500):
                    await btn.scroll_into_view_if_needed()
                    await btn.click()
                    log.info(f"  Butona tıklandı: {btn_sel}")
                    await asyncio.sleep(2.0)
                    break
            except Exception:
                pass

    card_els = await page.locator("a.card.news-card-horizontal").all()
    log.info(f"Toplam {len(card_els)} kart bulundu.")

    items = []
    for card in card_els:
        href = await card.get_attribute("href") or ""
        if href.startswith("//"):
            href = "https:" + href
        elif href.startswith("/"):
            href = "https://www.kahramanmaras.gov.tr" + href

        title  = await _loc_text(card, "h5.card-title")
        date   = await _loc_text(card, "span.cardDate")
        teaser = await _loc_text(card, "p.card-text")

        if href:
            items.append({"href": href, "title": title, "date": date, "teaser": teaser})

    return items


# ─── Detay sayfası ───────────────────────────────────────────────────────────

async def scrape_detail(page: Page, url: str) -> dict:
    ok = await safe_goto(page, url)
    if not ok:
        return {"detail_title": "", "full_text": ""}

    await asyncio.sleep(1.0)

    # ── Başlık ───────────────────────────────────────────────────────────────
    detail_title = ""
    for sel in ["h1 span", "h2 span", "h1", "h2"]:
        try:
            el = page.locator(sel).first
            if await el.count():
                detail_title = (await el.inner_text(timeout=2_000)).strip()
                if detail_title:
                    break
        except Exception:
            pass

    # ── İçerik alanı ─────────────────────────────────────────────────────────
    content_el = None
    for sel in [
        '[id^="wd2d929"] > div',   # Kahramanmaraş ID pattern
        '[id^="w9727509"] > div',
        '[id^="w6e94dc"] > div',
        '[id$="2"] > div',
        ".news-detail-content",
        ".haber-detay-content",
        "article .content",
        ".detail-content",
        "main .col-md-8",
        "main .col-lg-8",
    ]:
        try:
            el = page.locator(sel).first
            if await el.count():
                content_el = el
                break
        except Exception:
            pass

    # ── Paragrafları topla ───────────────────────────────────────────────────
    paragraphs = []

    if content_el:
        # Önce div > div yapısını dene
        inner_divs      = content_el.locator("div > div")
        inner_div_count = await inner_divs.count()

        if inner_div_count:
            for j in range(inner_div_count):
                t = (await inner_divs.nth(j).inner_text()).strip()
                if t and len(t) > 10:
                    paragraphs.append(t)
        else:
            # p > span yapısını dene
            spans      = content_el.locator("p span")
            span_count = await spans.count()
            if span_count:
                for j in range(span_count):
                    t = (await spans.nth(j).inner_text()).strip()
                    if t:
                        paragraphs.append(t)
            else:
                # Düz p etiketleri
                ps       = content_el.locator("p")
                ps_count = await ps.count()
                for j in range(ps_count):
                    t = (await ps.nth(j).inner_text()).strip()
                    if t:
                        paragraphs.append(t)

    # Son çare
    if not paragraphs:
        ps       = page.locator("main p, article p, .container p")
        ps_count = await ps.count()
        for j in range(ps_count):
            t = (await ps.nth(j).inner_text()).strip()
            if t and len(t) > 30:
                paragraphs.append(t)

    seen, clean = set(), []
    for p in paragraphs:
        if p not in seen:
            seen.add(p)
            clean.append(p)

    return {"detail_title": detail_title, "full_text": "\n\n".join(clean)}


# ─── Ana akış ────────────────────────────────────────────────────────────────

async def main():
    log.info("Kahramanmaraş scraper başlatılıyor…")
    start_time = datetime.now()

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=HEADLESS,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-sandbox",
                "--disable-dev-shm-usage",
            ],
        )
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/123.0.0.0 Safari/537.36"
            ),
            locale="tr-TR",
            timezone_id="Europe/Istanbul",
            viewport={"width": 1280, "height": 900},
            extra_http_headers={
                "Accept-Language": "tr-TR,tr;q=0.9,en-US;q=0.8",
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            },
        )
        await context.add_init_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )

        page = await context.new_page()
        cards = await collect_cards(page)

        fieldnames = ["date", "title", "teaser", "url", "detail_title", "full_text"]

        with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

            for i, card in enumerate(cards, 1):
                log.info(f"[{i}/{len(cards)}] {card['title'][:60]}…")
                await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

                detail = await scrape_detail(page, card["href"])

                writer.writerow({
                    "date"         : card["date"],
                    "title"        : card["title"],
                    "teaser"       : card["teaser"],
                    "url"          : card["href"],
                    "detail_title" : detail["detail_title"],
                    "full_text"    : detail["full_text"],
                })
                f.flush()

        await browser.close()

    elapsed = datetime.now() - start_time
    log.info(f"Tamamlandı! {len(cards)} haber — Süre: {elapsed}")
    log.info(f"Çıktı: {OUTPUT_FILE}")


# Jupyter'da:  await main()
# Terminalde:  python kahramanmaras_scraper.py
await main()

19:06:14  INFO  Kahramanmaraş scraper başlatılıyor…
19:06:15  INFO  Ana sayfa yükleniyor: https://www.kahramanmaras.gov.tr/haberler
19:06:24  INFO    Scroll 1/70 — kart: 10
19:06:24  INFO    Butona tıklandı: a:has-text("Daha Fazla")
19:06:28  INFO    Scroll 2/70 — kart: 20
19:06:28  INFO    Butona tıklandı: a:has-text("Daha Fazla")
19:06:32  INFO    Scroll 3/70 — kart: 30
19:06:32  INFO    Butona tıklandı: a:has-text("Daha Fazla")
19:06:36  INFO    Scroll 4/70 — kart: 40
19:06:36  INFO    Butona tıklandı: a:has-text("Daha Fazla")
19:06:40  INFO    Scroll 5/70 — kart: 50
19:06:40  INFO    Butona tıklandı: a:has-text("Daha Fazla")
19:06:44  INFO    Scroll 6/70 — kart: 60
19:06:45  INFO    Butona tıklandı: a:has-text("Daha Fazla")
19:06:49  INFO    Scroll 7/70 — kart: 70
19:06:49  INFO    Butona tıklandı: a:has-text("Daha Fazla")
19:06:53  INFO    Scroll 8/70 — kart: 80
19:06:53  INFO    Butona tıklandı: a:has-text("Daha Fazla")
19:06:57  INFO    Scroll 9/70 — kart: 90
19:06:57  INFO    B

In [4]:
!pip install requests beautifulsoup4 lxml pandas tqdm


In [19]:
# ─── BACKFILL HÜCRESİ — bunu Jupyter hücresine yapıştır ve çalıştır ──────────
# tccb_konusmalar_p1_p15.csv içindeki boş `date` kolonlarını doldurur.

import asyncio, re, shutil, random
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

INPUT_CSV  = Path("/Users/mehmetbagdinli/Desktop/deprem/tccb_konusmalar_p1_p15.csv")
BACKUP_CSV = INPUT_CSV.with_suffix(".backup.csv")

HEADLESS = True
DELAY_MIN, DELAY_MAX = 0.3, 0.8
NAV_TIMEOUT_MS = 30_000
SEL_TIMEOUT_MS = 8_000
DATE_SELECTOR  = "h6"

TURK_AYLAR = {
    "ocak":1,"şubat":2,"subat":2,"mart":3,"nisan":4,"mayıs":5,"mayis":5,
    "haziran":6,"temmuz":7,"ağustos":8,"agustos":8,"eylül":9,"eylul":9,
    "ekim":10,"kasım":11,"kasim":11,"aralık":12,"aralik":12,
}
RE_TR = re.compile(
    r"(\d{1,2})\s+(Ocak|Şubat|Subat|Mart|Nisan|Mayıs|Mayis|Haziran|"
    r"Temmuz|Ağustos|Agustos|Eylül|Eylul|Ekim|Kasım|Kasim|Aralık|Aralik)\s+(\d{4})",
    re.IGNORECASE)
RE_NUM = re.compile(r"(\d{1,2})[.\-/](\d{1,2})[.\-/](\d{4})")

def parse_to_iso(t):
    if not t: return None
    t = t.strip()
    m = RE_NUM.search(t)
    if m:
        d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if 1 <= d <= 31 and 1 <= mo <= 12:
            return f"{y:04d}-{mo:02d}-{d:02d}"
    m = RE_TR.search(t)
    if m:
        d, mo, y = int(m.group(1)), TURK_AYLAR[m.group(2).lower()], int(m.group(3))
        if 1 <= d <= 31 and 1 <= mo <= 12:
            return f"{y:04d}-{mo:02d}-{d:02d}"
    return None

async def main():
    if not INPUT_CSV.exists():
        raise FileNotFoundError(INPUT_CSV)
    if not BACKUP_CSV.exists():
        shutil.copy2(INPUT_CSV, BACKUP_CSV)
        print(f"✓ Yedek: {BACKUP_CSV}")

    df = pd.read_csv(INPUT_CSV)
    if "date_text" not in df.columns:
        df["date_text"] = pd.NA

    todo_mask = df["date"].isna() | (df["date"].astype(str).str.strip().isin(["","nan","NaN"]))
    todo_idx = df.index[todo_mask].tolist()
    print(f"→ {len(todo_idx)}/{len(df)} satır işlenecek.")
    if not todo_idx:
        print("Hepsi dolu, çıkılıyor."); return

    found = missing = errors = 0

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS,
            args=["--disable-blink-features=AutomationControlled","--no-sandbox","--disable-dev-shm-usage"])
        context = await browser.new_context(
            user_agent=("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"),
            locale="tr-TR", timezone_id="Europe/Istanbul",
            viewport={"width":1280,"height":900},
            extra_http_headers={"Accept-Language":"tr-TR,tr;q=0.9,en-US;q=0.8"})

        async def _block(route):
            if route.request.resource_type in {"image","media","font","stylesheet"}:
                await route.abort()
            else:
                await route.continue_()
        await context.route("**/*", _block)

        page = await context.new_page()
        page.set_default_timeout(NAV_TIMEOUT_MS)

        for i in tqdm(todo_idx, desc="Tarih çekiliyor"):
            url = df.at[i, "link"]
            if not isinstance(url, str) or not url.startswith("http"):
                errors += 1; continue
            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=NAV_TIMEOUT_MS)
                try:
                    await page.wait_for_selector(DATE_SELECTOR, timeout=SEL_TIMEOUT_MS)
                except PWTimeout:
                    pass
                loc = page.locator(DATE_SELECTOR).first
                raw = (await loc.inner_text()).strip() if await loc.count() > 0 else ""
                if raw:
                    iso = parse_to_iso(raw)
                    df.at[i, "date_text"] = raw
                    df.at[i, "date"]      = iso if iso else raw
                    found += 1
                else:
                    missing += 1
                    print(f"  ⚠ boş: {url}")
            except Exception as e:
                errors += 1
                print(f"  ✗ {type(e).__name__}: {url}")
            df.to_csv(INPUT_CSV, index=False, encoding="utf-8-sig")
            await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

        await browser.close()

    print(f"\n✓ Bitti.  Bulunan: {found}  Boş: {missing}  Hata: {errors}")

await main()

→ 600/600 satır işlenecek.


Tarih çekiliyor:   0%|          | 0/600 [00:00<?, ?it/s]

/var/folders/fq/hry5knqn05n0lrcxpj8fdw_r0000gn/T/ipykernel_6922/281162286.py:98: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '18.03.2026' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, "date_text"] = raw
/var/folders/fq/hry5knqn05n0lrcxpj8fdw_r0000gn/T/ipykernel_6922/281162286.py:99: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2026-03-18' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, "date"]      = iso if iso else raw



✓ Bitti.  Bulunan: 600  Boş: 0  Hata: 0


In [20]:
import pandas as pd
df = pd.read_csv("/Users/mehmetbagdinli/Desktop/deprem/tccb_konusmalar_p1_p15.csv")

print("Satır:", len(df))
print("Boş date:", df["date"].isna().sum())
print("\nİlk 5:")
print(df[["id", "title", "date", "date_text"]].head())
print("\nSon 5:")
print(df[["id", "title", "date", "date_text"]].tail())
print("\nTarih aralığı:", df["date"].min(), "→", df["date"].max())


Satır: 600
Boş date: 0

İlk 5:
       id                                              title        date  \
0  164376  Gazeteci ve Yazarlarla İftar Programı’nda Yapt...  2026-03-18   
1  164375  Hacı İbrahim Demir Camii Açılış Programı’nda Y...  2026-03-17   
2  164354  Kur’an-ı Kerim’i Güzel Okuma Yarışması Ödül Tö...  2026-03-16   
3  164346  14 Mart Tıp Bayramı Münasebetiyle İftar Progra...  2026-03-14   
4  164345  10. Milli İrade İftarı Programı’nda Yaptıkları...  2026-03-13   

    date_text  
0  18.03.2026  
1  17.03.2026  
2  16.03.2026  
3  14.03.2026  
4  13.03.2026  

Son 5:
         id                                              title        date  \
595  142731  Afyonkarahisar-Şuhut Yolu Açılış Töreni’nde Ya...  2023-02-02   
596  142693  Denizli Kadın İşçilerle Buluşma Programı’nda Y...  2023-01-28   
597  142689   Bilecik Gençlik Buluşması’nda Yaptıkları Konuşma  2023-01-27   
598  142687  Denizli Toplu Açılış Töreni’nde Yaptıkları Kon...  2023-01-28   
599  142686  Bilec

In [21]:
# ─── TANILAMA HÜCRESİ — 3 sitenin selektörlerini test et ─────────────────────
# Üç sitede de bir URL açar, tarih ve içerik için hangi seçicinin çalıştığını gösterir.
# ÇIKTIYI BANA YAPIŞTIR — buna göre backfill scriptlerini ayarlayacağım.

from playwright.async_api import async_playwright

TEST_URLS = {
    "hatay_belediye": "https://hatay.bel.tr/haberler/hataybotta-final-tamamlandi-oduller-sahiplerini-buldu",
    "adiyaman_valilik": "https://www.adiyaman.gov.tr/2025-yili-genel-degerlendirme-toplantisi",
    "kahramanmaras_valilik": "https://www.kahramanmaras.gov.tr/istiklal-madalyasinin-101-yil-donumu-torenle-kutlandi",
}

# Test edilecek seçiciler (tarih + içerik aday listeleri)
DATE_SELECTORS = [
    "h6", "time", ".tarih", ".date", ".cardDate", ".news-date",
    "[class*='date']", "[class*='tarih']",
    "meta[property='article:published_time']",
    "meta[name='date']", "meta[name='pubdate']",
]

CONTENT_SELECTORS = [
    "article", "main article",
    ".news-detail-content", ".haber-detay-content", ".detail-content",
    ".news-content", ".haber-icerik", ".content-area",
    "main .col-md-8", "main .col-lg-8",
    "[class*='detail']", "[class*='content']",
]

async def diagnose_site(name, url):
    print(f"\n{'='*70}\n  {name}\n  URL: {url}\n{'='*70}")
    
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"),
            locale="tr-TR", viewport={"width":1280,"height":900},
        )
        page = await context.new_page()
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=30_000)
            await page.wait_for_timeout(3000)
        except Exception as e:
            print(f"  ✗ Sayfa yüklenemedi: {e}")
            await browser.close()
            return
        
        # Cookie popup'unu kapat
        for sel in ["button:has-text('Kabul')", "button:has-text('Tamam')", "[class*='cookie'] button"]:
            try:
                if await page.locator(sel).count() > 0:
                    await page.locator(sel).first.click(timeout=1500)
                    await page.wait_for_timeout(500)
                    break
            except Exception: pass
        
        # ── Tarih seçici testi ──
        print("\n  --- TARİH ---")
        for sel in DATE_SELECTORS:
            try:
                cnt = await page.locator(sel).count()
                if cnt > 0:
                    txt = ""
                    if sel.startswith("meta"):
                        txt = await page.locator(sel).first.get_attribute("content") or ""
                    else:
                        txt = (await page.locator(sel).first.inner_text()).strip()
                    print(f"    {sel:50s} → {cnt} eşleşme  [{txt[:80]!r}]")
            except Exception: pass
        
        # ── İçerik seçici testi ──
        print("\n  --- İÇERİK (en uzun text 200 karakteri) ---")
        for sel in CONTENT_SELECTORS:
            try:
                cnt = await page.locator(sel).count()
                if cnt > 0:
                    txt = (await page.locator(sel).first.inner_text()).strip()
                    print(f"    {sel:40s} → {cnt} eşleşme, {len(txt)} char")
                    if len(txt) > 100:
                        print(f"        ↳ {txt[:200]!r}")
            except Exception: pass
        
        # ── Tüm <p> etiketleri ──
        ps = page.locator("p")
        cnt = await ps.count()
        print(f"\n  --- Toplam <p> sayısı: {cnt} ---")
        if cnt > 0:
            # En uzun 3 paragrafı göster
            ptexts = []
            for i in range(min(cnt, 60)):
                t = (await ps.nth(i).inner_text()).strip()
                if t: ptexts.append(t)
            ptexts.sort(key=len, reverse=True)
            for j, t in enumerate(ptexts[:3]):
                print(f"    [{j}] ({len(t)} char) {t[:200]!r}")
        
        # ── Sayfa title ──
        title = await page.title()
        print(f"\n  Sayfa başlığı: {title!r}")
        
        await browser.close()

for name, url in TEST_URLS.items():
    await diagnose_site(name, url)


  hatay_belediye
  URL: https://hatay.bel.tr/haberler/hataybotta-final-tamamlandi-oduller-sahiplerini-buldu

  --- TARİH ---

  --- İÇERİK (en uzun text 200 karakteri) ---
    .detail-content                          → 1 eşleşme, 2299 char
        ↳ '04.04.2026\nHATAYBOT’TA FİNAL TAMAMLANDI, ÖDÜLLER SAHİPLERİNİ BULDU\n\nHatay Valiliği, Hatay Büyükşehir Belediyesi (HBB) ve Hatay İl Milli Eğitim Müdürlüğü iş birliğiyle bu yıl ikincisi düzenlenen Hatay '
    [class*='detail']                        → 3 eşleşme, 2510 char
        ↳ '04.04.2026\nHATAYBOT’TA FİNAL TAMAMLANDI, ÖDÜLLER SAHİPLERİNİ BULDU\n\nHatay Valiliği, Hatay Büyükşehir Belediyesi (HBB) ve Hatay İl Milli Eğitim Müdürlüğü iş birliğiyle bu yıl ikincisi düzenlenen Hatay '
    [class*='content']                       → 8 eşleşme, 0 char

  --- Toplam <p> sayısı: 12 ---
    [0] (307 char) 'Yarışma sonucunda Mucit kategorisinde dereceye giren projeler de netleşti. Buna göre, HBB Hatay Bilim Merkezi bünyesinde geliştirilen “Akıllı

In [22]:
# ─── HÜCRE 1: HATAY BELEDİYE — TARİH BACKFILL ────────────────────────────────
# 2408 satırın `date` kolonu boş. Her URL'in detay sayfasından `.detail-content`
# içindeki ilk DD.MM.YYYY tarihini çekip CSV'ye yazar.
#
# Süre: ~60 dakika (Ctrl+C ile durdurursan kaldığı yerden devam eder)

import asyncio, re, shutil, random
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

INPUT_CSV  = Path("/Users/mehmetbagdinli/Desktop/deprem/hatay_belediye_haberler.csv")
BACKUP_CSV = INPUT_CSV.with_suffix(".backup.csv")

HEADLESS = True
DELAY_MIN, DELAY_MAX = 0.3, 0.7
NAV_TIMEOUT_MS = 30_000
SEL_TIMEOUT_MS = 8_000

DETAIL_SEL = ".detail-content"
RE_DATE    = re.compile(r"\b(\d{1,2})\.(\d{1,2})\.(\d{4})\b")

def parse_to_iso(text):
    """'04.04.2026' → '2026-04-04'. Olmazsa None."""
    if not text: return None
    m = RE_DATE.search(text)
    if m:
        d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if 1 <= d <= 31 and 1 <= mo <= 12:
            return f"{y:04d}-{mo:02d}-{d:02d}"
    return None

async def main():
    if not INPUT_CSV.exists():
        raise FileNotFoundError(INPUT_CSV)
    if not BACKUP_CSV.exists():
        shutil.copy2(INPUT_CSV, BACKUP_CSV)
        print(f"✓ Yedek: {BACKUP_CSV}")

    df = pd.read_csv(INPUT_CSV)
    if "date_text" not in df.columns:
        df["date_text"] = pd.NA

    todo_mask = df["date"].isna() | (df["date"].astype(str).str.strip().isin(["","nan","NaN"]))
    todo_idx = df.index[todo_mask].tolist()
    print(f"→ {len(todo_idx)}/{len(df)} satır işlenecek.")
    if not todo_idx:
        print("Hepsi dolu, çıkılıyor."); return

    found = missing = errors = 0

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS,
            args=["--disable-blink-features=AutomationControlled","--no-sandbox","--disable-dev-shm-usage"])
        context = await browser.new_context(
            user_agent=("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"),
            locale="tr-TR", timezone_id="Europe/Istanbul",
            viewport={"width":1280,"height":900},
            extra_http_headers={"Accept-Language":"tr-TR,tr;q=0.9,en-US;q=0.8"})
        # Görsel/CSS bloğu — hızlandırma
        async def _block(route):
            if route.request.resource_type in {"image","media","font","stylesheet"}:
                await route.abort()
            else:
                await route.continue_()
        await context.route("**/*", _block)

        page = await context.new_page()
        page.set_default_timeout(NAV_TIMEOUT_MS)

        for i in tqdm(todo_idx, desc="Hatay Bld tarih"):
            url = df.at[i, "url"]
            if not isinstance(url, str) or not url.startswith("http"):
                errors += 1; continue
            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=NAV_TIMEOUT_MS)
                try:
                    await page.wait_for_selector(DETAIL_SEL, timeout=SEL_TIMEOUT_MS)
                except PWTimeout:
                    pass
                loc = page.locator(DETAIL_SEL).first
                raw = (await loc.inner_text()).strip() if await loc.count() > 0 else ""
                # İlk 100 karakterden tarih çıkar
                iso = parse_to_iso(raw[:100])
                if iso:
                    df.at[i, "date"]      = iso
                    df.at[i, "date_text"] = raw[:30].split("\n")[0].strip()
                    found += 1
                else:
                    missing += 1
                    if missing <= 5:
                        print(f"  ⚠ tarih bulunamadı: {url}")
            except Exception as e:
                errors += 1
                if errors <= 5:
                    print(f"  ✗ {type(e).__name__}: {url}")
            df.to_csv(INPUT_CSV, index=False, encoding="utf-8-sig")
            await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

        await browser.close()
    print(f"\n✓ Bitti.  Bulunan: {found}  Boş: {missing}  Hata: {errors}")

await main()

✓ Yedek: /Users/mehmetbagdinli/Desktop/deprem/hatay_belediye_haberler.backup.csv
→ 2408/2408 satır işlenecek.


Hatay Bld tarih:   0%|          | 0/2408 [00:00<?, ?it/s]

/var/folders/fq/hry5knqn05n0lrcxpj8fdw_r0000gn/T/ipykernel_6922/99229532.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2026-04-04' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, "date"]      = iso


  ⚠ tarih bulunamadı: https://hatay.bel.tr/haberler/alo-153-bir-telefonla-tum-hataya-cozum-sunuyor
  ⚠ tarih bulunamadı: https://hatay.bel.tr/News/a8f4d0fb-e2c4-48ab-93e9-da78b1d043a7

✓ Bitti.  Bulunan: 2406  Boş: 2  Hata: 0


In [23]:
# ─── HÜCRE 3: KAHRAMANMARAŞ VALİLİĞİ — FULL_TEXT BACKFILL ────────────────────
# 710 satırın 704'ünde full_text boş. Adıyaman ile aynı yöntem (aynı altyapı:
# T.C. İçişleri Bakanlığı şablonu) — [class*='detail']'den metin çekilir.
#
# Süre: ~15-20 dakika

import asyncio, re, shutil, random
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

INPUT_CSV  = Path("/Users/mehmetbagdinli/Desktop/deprem/kahramanmaras_haberler.csv")
BACKUP_CSV = INPUT_CSV.with_suffix(".backup.csv")

HEADLESS = True
DELAY_MIN, DELAY_MAX = 0.4, 0.9
NAV_TIMEOUT_MS = 30_000
SEL_TIMEOUT_MS = 8_000

DETAIL_SEL  = "[class*='detail']"
RE_DATELINE = re.compile(r"^\s*\d{1,2}\.\d{1,2}\.\d{4}\s*$", re.MULTILINE)

def extract_body(raw_text):
    if not raw_text:
        return ""
    m = RE_DATELINE.search(raw_text)
    if m:
        body = raw_text[m.end():]
    else:
        lines = raw_text.split("\n", 2)
        body = lines[2] if len(lines) >= 3 else raw_text
    body = body.replace("\xa0", " ")
    body = re.sub(r"\n{3,}", "\n\n", body)
    body = re.sub(r"[ \t]+", " ", body)
    return body.strip()

def extract_title(raw_text):
    if not raw_text: return ""
    return raw_text.split("\n", 1)[0].strip()

async def main():
    if not INPUT_CSV.exists():
        raise FileNotFoundError(INPUT_CSV)
    if not BACKUP_CSV.exists():
        shutil.copy2(INPUT_CSV, BACKUP_CSV)
        print(f"✓ Yedek: {BACKUP_CSV}")

    df = pd.read_csv(INPUT_CSV)
    todo_mask = df["full_text"].isna() | (df["full_text"].astype(str).str.strip().isin(["","nan","NaN"]))
    todo_idx = df.index[todo_mask].tolist()
    print(f"→ {len(todo_idx)}/{len(df)} satır işlenecek.")
    if not todo_idx:
        print("Hepsi dolu, çıkılıyor."); return

    found = missing = errors = 0

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS,
            args=["--disable-blink-features=AutomationControlled","--no-sandbox","--disable-dev-shm-usage"])
        context = await browser.new_context(
            user_agent=("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"),
            locale="tr-TR", timezone_id="Europe/Istanbul",
            viewport={"width":1280,"height":900},
            extra_http_headers={"Accept-Language":"tr-TR,tr;q=0.9,en-US;q=0.8"})
        async def _block(route):
            if route.request.resource_type in {"image","media","font","stylesheet"}:
                await route.abort()
            else:
                await route.continue_()
        await context.route("**/*", _block)

        page = await context.new_page()
        page.set_default_timeout(NAV_TIMEOUT_MS)

        for i in tqdm(todo_idx, desc="Kahramanmaraş valilik"):
            url = df.at[i, "url"]
            if not isinstance(url, str) or not url.startswith("http"):
                errors += 1; continue
            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=NAV_TIMEOUT_MS)
                try:
                    await page.wait_for_selector(DETAIL_SEL, timeout=SEL_TIMEOUT_MS)
                except PWTimeout:
                    pass
                loc = page.locator(DETAIL_SEL).first
                raw = (await loc.inner_text()).strip() if await loc.count() > 0 else ""
                body = extract_body(raw)
                if body:
                    df.at[i, "full_text"]    = body
                    df.at[i, "detail_title"] = extract_title(raw)
                    found += 1
                else:
                    missing += 1
                    if missing <= 5:
                        print(f"  ⚠ metin bulunamadı: {url}")
            except Exception as e:
                errors += 1
                if errors <= 5:
                    print(f"  ✗ {type(e).__name__}: {url}")
            df.to_csv(INPUT_CSV, index=False, encoding="utf-8-sig")
            await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

        await browser.close()
    print(f"\n✓ Bitti.  Bulunan: {found}  Boş: {missing}  Hata: {errors}")

await main()

✓ Yedek: /Users/mehmetbagdinli/Desktop/deprem/kahramanmaras_haberler.backup.csv
→ 704/710 satır işlenecek.


Kahramanmaraş valilik:   0%|          | 0/704 [00:00<?, ?it/s]


✓ Bitti.  Bulunan: 704  Boş: 0  Hata: 0


In [24]:
# ─── HÜCRE 2: ADIYAMAN VALİLİĞİ — FULL_TEXT BACKFILL ─────────────────────────
# 70 satırın 41'inde full_text boş (scraper çerez popup'una takılmış).
# Bu hücre boş olanlara gidip [class*='detail'] içeriğinden gerçek metni çeker.
#
# Süre: ~1-2 dakika

import asyncio, re, shutil, random
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

INPUT_CSV  = Path("/Users/mehmetbagdinli/Desktop/deprem/adiyaman_haberler.csv")
BACKUP_CSV = INPUT_CSV.with_suffix(".backup.csv")

HEADLESS = True
DELAY_MIN, DELAY_MAX = 0.4, 0.9
NAV_TIMEOUT_MS = 30_000
SEL_TIMEOUT_MS = 8_000

DETAIL_SEL = "[class*='detail']"
# detail içeriği formatı: "BAŞLIK\nDD.MM.YYYY\n[\xa0\n]?İçerik..."
# Tarih satırını bulup ondan sonrasını metin olarak alacağız.
RE_DATELINE = re.compile(r"^\s*\d{1,2}\.\d{1,2}\.\d{4}\s*$", re.MULTILINE)

def extract_body(raw_text):
    """detail-content tam metnini al, başlık+tarih satırlarını çıkar, geri kalanı döndür."""
    if not raw_text:
        return ""
    # Tarih satırını bul
    m = RE_DATELINE.search(raw_text)
    if m:
        body = raw_text[m.end():]
    else:
        # Tarih bulamadıysak ilk iki satırı atla (başlık + tarih)
        lines = raw_text.split("\n", 2)
        body = lines[2] if len(lines) >= 3 else raw_text
    # Temizle
    body = body.replace("\xa0", " ")
    body = re.sub(r"\n{3,}", "\n\n", body)
    body = re.sub(r"[ \t]+", " ", body)
    return body.strip()

def extract_title(raw_text):
    """detail-content'in ilk satırı = başlık"""
    if not raw_text: return ""
    return raw_text.split("\n", 1)[0].strip()

async def main():
    if not INPUT_CSV.exists():
        raise FileNotFoundError(INPUT_CSV)
    if not BACKUP_CSV.exists():
        shutil.copy2(INPUT_CSV, BACKUP_CSV)
        print(f"✓ Yedek: {BACKUP_CSV}")

    df = pd.read_csv(INPUT_CSV)
    # full_text boş olanlar
    todo_mask = df["full_text"].isna() | (df["full_text"].astype(str).str.strip().isin(["","nan","NaN"]))
    todo_idx = df.index[todo_mask].tolist()
    print(f"→ {len(todo_idx)}/{len(df)} satır işlenecek (full_text boş olanlar).")
    if not todo_idx:
        print("Hepsi dolu, çıkılıyor."); return

    found = missing = errors = 0

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS,
            args=["--disable-blink-features=AutomationControlled","--no-sandbox","--disable-dev-shm-usage"])
        context = await browser.new_context(
            user_agent=("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"),
            locale="tr-TR", timezone_id="Europe/Istanbul",
            viewport={"width":1280,"height":900},
            extra_http_headers={"Accept-Language":"tr-TR,tr;q=0.9,en-US;q=0.8"})
        async def _block(route):
            if route.request.resource_type in {"image","media","font","stylesheet"}:
                await route.abort()
            else:
                await route.continue_()
        await context.route("**/*", _block)

        page = await context.new_page()
        page.set_default_timeout(NAV_TIMEOUT_MS)

        for i in tqdm(todo_idx, desc="Adıyaman valilik"):
            url = df.at[i, "url"]
            if not isinstance(url, str) or not url.startswith("http"):
                errors += 1; continue
            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=NAV_TIMEOUT_MS)
                try:
                    await page.wait_for_selector(DETAIL_SEL, timeout=SEL_TIMEOUT_MS)
                except PWTimeout:
                    pass
                loc = page.locator(DETAIL_SEL).first
                raw = (await loc.inner_text()).strip() if await loc.count() > 0 else ""
                body = extract_body(raw)
                if body:
                    df.at[i, "full_text"]    = body
                    df.at[i, "detail_title"] = extract_title(raw)
                    found += 1
                else:
                    missing += 1
                    if missing <= 5:
                        print(f"  ⚠ metin bulunamadı: {url}")
            except Exception as e:
                errors += 1
                if errors <= 5:
                    print(f"  ✗ {type(e).__name__}: {url}")
            df.to_csv(INPUT_CSV, index=False, encoding="utf-8-sig")
            await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

        await browser.close()
    print(f"\n✓ Bitti.  Bulunan: {found}  Boş: {missing}  Hata: {errors}")

await main()

✓ Yedek: /Users/mehmetbagdinli/Desktop/deprem/adiyaman_haberler.backup.csv
→ 41/70 satır işlenecek (full_text boş olanlar).


Adıyaman valilik:   0%|          | 0/41 [00:00<?, ?it/s]


✓ Bitti.  Bulunan: 41  Boş: 0  Hata: 0
